<a href="https://colab.research.google.com/github/UzunDemir/maib_agent_1/blob/main/maib_bot_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Пример решения задачи чат-бота

* Без FAISS (Если не много документов)
* С FAISS
* BLOOM как вариант мультиязычной LLM

По вчерашнему собеседованию требовалось: модель должна возвращать 10 самых релевантных документов из которых выбрать топ-3 для контекста на подачу в LLM, использовать Qwen.

In [1]:
# если имеем, то GPU
! pip install faiss-cpu


## Без FAISS (Если не много документов) до 1000, например

In [2]:
from sentence_transformers import SentenceTransformer, CrossEncoder, util
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Документы
documents = [
    "At MAIB, we offer a dynamic workplace for professionals in data and technology.",
    "Our HR chatbot helps candidates find relevant job openings and apply with ease.",
    "The company culture at MAIB encourages innovation, responsibility, and teamwork.",
    "You can check your application status anytime using our career portal.",
    "Our benefits include health insurance, flexible hours, and growth opportunities.",
    "We value diversity and aim to provide an inclusive environment for all employees.",
    "The recruitment process includes screening, interviews, and an assessment task.",
    "HR uses AI tools to screen resumes and schedule interviews automatically.",
    "MAIB is the leading commercial bank in Moldova, focusing on digital transformation.",
    "Join MAIB to grow your career in a supportive and forward-thinking environment.",
    "The chatbot supports queries about benefits, vacation policy, and internal jobs.",
    "Resumes are reviewed weekly by HR managers and AI-assisted systems.",
    "Applicants can upload their resumes directly to the hiring dashboard.",
    "MAIB provides internal mobility for employees to explore different departments.",
    "We regularly host webinars and career workshops for students and professionals.",
]



In [3]:
# Загружаем модели эмбеддингов и rerank
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
rerank_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')



/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
# Векторизация документов
doc_embeddings = embedding_model.encode(documents, convert_to_tensor=True)



In [6]:
# Запрос пользователя пока английский...
query = "What is the recruitment process at MAIB?"
query_embedding = embedding_model.encode(query, convert_to_tensor=True)



In [8]:
# Извлекаем топ-10 по косинусному сходству
# Можно и Дот-продукт (скалярное произведение)
cos_scores = util.cos_sim(query_embedding, doc_embeddings)[0]
top_10_idx = np.argpartition(-cos_scores.cpu(), range(10))[:10]
top_10_docs = [documents[i] for i in top_10_idx.cpu()]



In [9]:
# Переранжируем топ-10 с CrossEncoder
rerank_input = [[query, doc] for doc in top_10_docs]
rerank_scores = rerank_model.predict(rerank_input)
reranked_indices = np.argsort(rerank_scores)[::-1]
top_docs = [top_10_docs[i] for i in reranked_indices][:3]  # Возьмём топ-3 для контекста



In [11]:
# Формируем prompt для Qwen в формате чата
context_text = "\n".join([f"{i+1}. {doc}" for i, doc in enumerate(top_docs)])
prompt = f"""<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
{query} Here is some context:
{context_text}
<|im_end|>
<|im_start|>assistant
"""


In [12]:
# Загружаем токенайзер и модель Qwen (можно и нужно большую модель)
model_name = "Qwen/Qwen1.5-0.5B-Chat"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True)
inputs = tokenizer(prompt, return_tensors="pt")


In [13]:
import time


with torch.no_grad():
    start_time = time.time()
    output = model.generate(
        **inputs,
        max_new_tokens=150,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )
    end_time = time.time()

answer = tokenizer.decode(output[0], skip_special_tokens=True)

print(f"\n⏳ Время генерации: {end_time - start_time:.3f} секунд\n")

# Выводим ответ после маркера assistant
print("\n📌 Ответ модели:\n")
print(answer.split("<|im_start|>assistant")[-1].strip())



⏳ Время генерации: 44.609 секунд


📌 Ответ модели:

system
You are a helpful assistant.
user
What is the recruitment process at MAIB? Here is some context:
1. The recruitment process includes screening, interviews, and an assessment task.
2. MAIB provides internal mobility for employees to explore different departments.
3. At MAIB, we offer a dynamic workplace for professionals in data and technology.

assistant
The recruitment process at MAIB involves several steps that can help determine if they are a good fit for the company's needs. Here are some key points:

  1. screening: This stage of the process involves evaluating the candidate's qualifications and experience. This may include a combination of physical tests and computer-based skills tests. It may also involve a review of resumes and cover letters.
  2. Interviews: During the interview, the employer assesses the candidate's communication skills, problem-solving abilities, and work ethic. They may also ask questions about the

## FAISS

In [16]:
from sentence_transformers import SentenceTransformer, CrossEncoder, util
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import faiss


# Загружаем модели эмбеддингов и rerank
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
rerank_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# Векторизация документов
doc_embeddings = embedding_model.encode(documents, convert_to_numpy=True, normalize_embeddings=True)  # здесь нормализир для косинуса

# Создаём FAISS индекс для поиска по cosin
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # после нормализации — косинусное сходство
index.add(doc_embeddings)  # эмбеддинги документов

# Запрос пользователя
query = "What is the recruitment process at MAIB?"
query_embedding = embedding_model.encode(query, convert_to_numpy=True, normalize_embeddings=True)

# Поиск топ-10 релевантных документов через FAISS
top_k = 10
D, I = index.search(query_embedding.reshape(1, -1), top_k)  # D — значения сходства, I — индексы документов
top_10_docs = [documents[i] for i in I[0]]

# Переранжируем топ-10 с CrossEncoder
rerank_input = [[query, doc] for doc in top_10_docs]
rerank_scores = rerank_model.predict(rerank_input)
reranked_indices = np.argsort(rerank_scores)[::-1]
top_docs = [top_10_docs[i] for i in reranked_indices][:3]  # топ-3 для контекста

# Формируем prompt для Qwen в формате чата
context_text = "\n".join([f"{i+1}. {doc}" for i, doc in enumerate(top_docs)])
prompt = f"""<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
{query} Here is some context:
{context_text}
<|im_end|>
<|im_start|>assistant
"""

# Загружаем токенизатор и модель Qwen
model_name = "Qwen/Qwen1.5-0.5B-Chat"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True)

inputs = tokenizer(prompt, return_tensors="pt")

# # Генерируем ответ
# with torch.no_grad():
#     output = model.generate(
#         **inputs,
#         max_new_tokens=150,
#         eos_token_id=tokenizer.eos_token_id,
#         pad_token_id=tokenizer.eos_token_id,
#     )

# answer = tokenizer.decode(output[0], skip_special_tokens=True)

# # Выводим ответ модели после маркера assistant
# print("\n📌 Ответ модели:\n")
# print(answer.split("<|im_start|>assistant")[-1].strip())


In [17]:
with torch.no_grad():
    start_time = time.time()
    output = model.generate(
        **inputs,
        max_new_tokens=150,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )
    end_time = time.time()

answer = tokenizer.decode(output[0], skip_special_tokens=True)

print(f"\n⏳ Время генерации: {end_time - start_time:.3f} секунд\n")

# Выводим ответ после маркера assistant
print("\n📌 Ответ модели:\n")
print(answer.split("<|im_start|>assistant")[-1].strip())




⏳ Время генерации: 52.978 секунд


📌 Ответ модели:

system
You are a helpful assistant.
user
What is the recruitment process at MAIB? Here is some context:
1. The recruitment process includes screening, interviews, and an assessment task.
2. MAIB provides internal mobility for employees to explore different departments.
3. At MAIB, we offer a dynamic workplace for professionals in data and technology.

assistant
The recruitment process at MAIB involves several steps, including:

  1. Screening: Maib reviews resumes and applications to identify qualified candidates who meet its requirements.
  2. Interviews: Interviewers assess candidates' skills, experience, and qualifications based on the job requirements.
  3. Assessment Task: The hiring manager then evaluates the candidate's performance and qualifications against the job description, and assigns them the position.

MAIB encourages its employees to explore different departments within the organization by offering internal mobility, 

## BLOOM как вариант мультиязычной LLM

In [49]:
from sentence_transformers import SentenceTransformer, CrossEncoder, util
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import faiss

# 1. Документы (русский + румынский)
documents = [
    "MAIB oferă un mediu dinamic și orientat spre viitor, perfect pentru profesioniștii pasionați de tehnologie și analiza datelor. Aici vei lucra cu cele mai recente instrumente și metodologii, într-un ecosistem care încurajează învățarea continuă.",
    "Chatbotul nostru HR, asistentul virtual Alex, te poate ghida pas cu pas prin procesul de aplicare. El răspunde la întrebări frecvente, verifică compatibilitatea CV-ului tău cu rolurile disponibile și chiar programează interviuri în funcție de disponibilitatea ta.",
    "Cultura MAIB este una deschisă, inovatoare și orientată spre rezultate. Promovăm colaborarea între departamente, iar fiecare angajat este încurajat să își expun ideile - de la stagiari până la manageri.",
    "Portalul nostru de carieră oferă transparență totală: poți verifica în timp real stadiul aplicației, descărca documente necesare sau primi notificări imediate când apare un nou post potrivit profilului tău.",
    "Pachetul nostru de beneficii include nu doar asigurări medicale premium și bonusuri de performanță, ci și opțiuni personalizabile: de la abonamente la săli de fitness până la bugete pentru învățare sau well-being.",
    "Diversitatea și incluziunea sunt la centrul valorilor noastre. Avem politici clare pentru egalitate de șanse, grupuri de susținere pentru angajați și programe de conștientizare lunară pe teme sociale importante.",
    "Procesul de recrutare este transparent și eficient: după screening-ul CV-urilor (automatizat prin AI), urmează un interviu telefonic, o discuție tehnică cu echipa și un test practic relevant pentru rolul dorit.",
    "Folosim tehnologii AI nu pentru a înlocui deciziile umane, ci pentru a optimiza procesele. Sistemul nostru analizează CV-urile pentru a identifica rapid potrivirile perfecte, economisind timp atât ție, cât și echipelor noastre.",
    "MAIB nu este doar o bancă - suntem un hub de inovație financiară. Colaborăm cu startup-uri fintech, testăm soluții blockchain și reinventăm modul în care clienții interacționează cu banii.",
    "Alătură-te MAIB pentru a construi o carieră cu impact real. Oferim nu doar un loc de muncă, ci un spațiu unde poți crește profesional, în echipa unor oameni talentați și deschiși.",
    "În primele 3 luni, beneficiezi de un program structurat de onboarding, cu training-uri practice, sesiuni one-on-one cu mentorii tăi și acces la o bibliotecă digitală cu resurse exclusive.",
    "Flexibilitatea este cheia: alegi între lucrul hibrid (2 zile acasă/3 la birou) sau full remote pentru anumite roluri, iar programul de lucru poate fi ajustat în funcție de nevoile personale.",
    "Organizăm săptămânal MAIB Tech Talks, unde invitați din industrie împărtășesc tendințe globale, iar angajații prezintă proiecte interne - o oportunitate fantastică de networking și învățare.",
    "Dacă recomanzi un prieten pentru un post deschis și este angajat, primești un bonus generos - până la 20% din salariul său lunar, în funcție de nivelul poziției.",
    "Pentru pasionații de tehnologie, avem MAIB Labs - un departament dedicat prototipării rapide, unde poți experimenta cu AI, big data sau instrumente financiare inovatoare înainte de lansare.",
    "Well-being-ul tău contează: oferim terapie gratuită, masaje la birou, și evenimente lunare dedicate sănătății mentale. Suntem prima bancă din Moldova cu un program dedicat burnout prevention.",
    "După primul an, poți opta pentru o rotație internă - schimbarea temporară a departamentului pentru a învăța noi skill-uri și a explora alte arii de interes.",
    "MAIB susține comunitatea prin proiecte de responsabilitate socială. Angajații primesc 2 zile plătite pe an pentru voluntariat și pot vota ce inițiative să sprijinim financiar.",
    "Dacă ești student, programul nostru de internship plătit (3-6 luni) îți oferă experiență practică, certificare și șansa de a fi angajat full-time după absolvire.",
    "Birourile noastre sunt spațioase, moderne și eco-friendly, cu zone de relaxare, bucătării bine aprovizionate și chiar un teren de ping-pong pentru pauzele creative.",
    "Procesul nostru de selecție durează în medie 2-3 săptămâni de la prima aplicare până la ofertă, în funcție de complexitatea rolului.",
    "Pentru posturile executive, procesul poate include și un assessment center cu simulări de business cases relevante pentru industrie.",
    "Toți candidații primesc feedback detaliat după fiecare etapă a interviului, chiar dacă nu sunt selectați.",
    "Acceptăm aplicații și în limba rusă sau engleză, iar interviurile pot fi conduse în oricare dintre aceste limbi la cerere.",
    "Nu există o limită de vârstă pentru aplicare - evaluăm competențele și experiența, indiferent de anul nașterii.",
    "Pentru persoanele cu dizabilități, adaptăm procesul de recrutare și oferim asistență tehnică pe parcursul interviurilor.",
    "Dacă ai fost respins la un interviu, poți reaplica după 6 luni, iar dosarul tău rămâne în baza noastră de date.",
    "Confidențialitatea este garantată - CV-ul tău va fi văzut doar de echipa HR și managerul direct pentru postul respectiv.",
    "Organizăm periodic zile de carieră virtuale unde poți interacționa direct cu reprezentanți ai departamentelor noastre.",
    "Poți solicita o copie a tuturor datelor pe care le deținem despre tine în procesul de recrutare, conform GDPR.",
    "Pentru posturile tehnice, procesul include adesea un challenge practic pe care îl poți completa acasă în 72 de ore.",
    "Nu practicăm întrebări personale despre statutul marital, planurile de familie sau apartenența religioasă.",
    "Oferim sesiuni de pregătire gratuite pentru interviuri candidaților care trec de prima etapă de screening.",
    "Dacă locuiești în alt oraș, acoperim cheltuielile de transport și cazare pentru interviurile finale la sediul central.",
    "Programul nostru de referal acordă bonusuri atât angajatului care recomandă, cât și candidatului acceptat.",
    "Pentru rolurile de leadership, folosim instrumente de evaluare psihometrică pentru a identifica potențialul de conducere.",
    "Poți solicita modificări la datele tale de contact sau retragerea aplicației în orice moment prin portalul de carieră.",
    "La MAIB, procesul de onboarding include o perioadă de acomodare de 3 luni cu sarcini progresive, pentru a te integra perfect în echipă.",
    "Toți noii angajați primesc un buddy - un coleg experimentat care te ajută să înțelegi cultura organizațională și procesele interne.",
    "Evaluările de performanță au loc trimestrial, cu focus pe dezvoltarea profesională și stabilirea unor obiective SMART.",
    "Programul nostru de mentoring te conectează cu lideri din organizație care te pot ghida în dezvoltarea carierei pe termen lung.",
    "Oferim posibilitatea de job shadowing - poți petrece o zi cu un coleg din alt departament pentru a înțelege mai bine activitatea acestuia.",
    "Pentru angajații care doresc să se reprofileze, oferim cursuri de recalificare profesională plătite de companie.",
    "Sistemul nostru de reward include nu doar bonusuri financiare, ci și premii nefinanciare ca zile libere suplimentare sau training-uri exclusive.",
    "La fiecare 2 ani, angajații pot beneficia de un sabatic plătit de până la 3 luni pentru proiecte personale de dezvoltare.",
    "Organizăm anual un Career Day intern unde poți explora oportunități în alte departamente și discuta direct cu managerii acestora.",
    "Programul de succesiune ne permite să identificăm și să dezvoltăm viitori lideri din interiorul organizației.",
    "Dacă pleci din companie, menținem legătura prin programul nostru Alumni și îți oferim prioritate la viitoare angajări.",
    "Pentru angajații cu peste 5 ani vechime, oferim programe speciale de retention care includ beneficii personalizate.",
    "Oferim sprijin pentru relocare atât pentru angajații din alte orașe, cât și pentru cei care doresc să lucreze din străinătate.",
    "Programul de wellness corporativ include check-up-uri medicale anuale gratuite și consiliere nutrițională personalizată.",
    "La MAIB credem în work-life integration - poți personaliza programul de lucru în funcție de nevoile tale personale și de familie.",
    "Pentru femeile care se întorc de la concediul de maternitate, oferim un program special de reintegrare progresivă.",
    "Organizăm quarterly feedback sessions unde poți discuta direct cu top managementul despre direcția companiei.",
    "Sistemul de internal mobility te încurajează să aplici pentru posturi vacante în alte departamente după minimum 1 an în funcție.",
    "Oferim burse pentru copiii angajaților care demonstrează excelență academică în domenii relevante pentru bancă.",
    "Programul de volunteering corporativ permite angajaților să inițieze propriile proiecte sociale cu sprijin financiar de la MAIB.",
    "Pentru angajații aproape de pensionare, oferim programe de tranziție care includ training-uri pentru a doua carieră.",
    "Clubul MAIB pentru angajați organizează lunar activități culturale, sportive și de team building în întreaga țară.",
    "Dacă ai o idee inovatoare, poți aplica pentru bugetul nostru de innovation grant pentru a o transforma în realitate.",
    "Oferim training-uri de leadership la toate nivelurile, de la team leader până la top management.",
    "Programul de digital literacy ajută angajații mai puțin familiarizați cu tehnologia să se adapteze la noile instrumente.",
    "La MAIB, 30% din posturile de conducere sunt ocupate prin promovare internă, evidențiind oportunitățile de creștere.",
    "Organizăm anual un Employee Experience Survey pentru a măsura satisfacția angajaților și a implementa îmbunătățiri.",
    "Pentru angajații care doresc să studieze, oferim programe flexibile și suport financiar pentru învățământul superior.",
    "Sistemul de cunoaștere între generații (GenZ-Boomers) încurajează schimbul de experiență între angajații de diferite vârste.",
    "Oferim posibilitatea de a lucra din spații de coworking în alte orașe pentru angajații cu regim remote.",
    "Programul de language improvement acoperă costurile cursurilor de limbă străină relevante pentru activitatea profesională.",
    "La MAIB, 85% din posturile vacante sunt publicate intern înainte de a fi aduse pe piață, oferind angajaților prima șansă.",
    "Organizăm hackathoane anuale unde angajații din toate departamentele pot colabora la soluții inovatoare pentru bancă.",
    "Pentru persoanele cu performanțe excepționale, oferim oportunități de reprezentare a băncii la conferințe internaționale.",
    "Programul de stress management include workshop-uri lunare și resurse pentru a gestiona mai bine presiunile profesionale.",
    "Oferim consultanță gratuită pentru probleme legale sau financiare personale prin programul nostru de asistență angajați.",
    "La MAIB, fiecare angajat are un plan individual de dezvoltare (IDP) actualizat anual împreună cu managerul direct.",
    "Organizăm turnee regulate ale tuturor sucursalelor pentru ca angajații să înțeleagă mai bine operațiunile din teritoriu.",
    "Programul de coaching executive este disponibil pentru toți angajații care doresc să își dezvolte abilități de top management."
]

# 2. Загружаем модели эмбеддингов и rerank
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
rerank_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# 3. Векторизация и нормализация для FAISS
doc_embeddings = embedding_model.encode(documents, convert_to_numpy=True, normalize_embeddings=True)

# 4. Создаем FAISS индекс для поиска
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(doc_embeddings)

# 5. Запрос пользователя (можно менять)
query = "Care este procesul de recrutare la MAIB?"  # Румынский
# query = "Каков процесс найма в MAIB?"  # Русский

query_embedding = embedding_model.encode(query, convert_to_numpy=True, normalize_embeddings=True)

# 6. Поиск топ-10 документов с FAISS
top_k = 10
D, I = index.search(query_embedding.reshape(1, -1), top_k)
top_10_docs = [documents[i] for i in I[0]]

# 7. Переранжировка с CrossEncoder
rerank_input = [[query, doc] for doc in top_10_docs]
rerank_scores = rerank_model.predict(rerank_input)
reranked_indices = np.argsort(rerank_scores)[::-1]
top_docs = [top_10_docs[i] for i in reranked_indices][:5]

# 8. Формируем prompt для BLOOM
context_text = "\n".join([f"{i+1}. {doc}" for i, doc in enumerate(top_docs)])

if query.strip().startswith("Care"):  # простой детектор языка (можно улучшить)
    prompt = f"Întrebare: {query}\nContext:\n{context_text}\nRăspuns:"
else:
    prompt = f"Вопрос: {query}\nКонтекст:\n{context_text}\nОтвет:"

# 9. Загружаем токенизатор и модель BLOOM
model_name = "bigscience/bloom-560m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# 10. Токенизация prompt
inputs = tokenizer(prompt, return_tensors="pt")

# 11. Генерация ответа
with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.5,
        top_p=0.9,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

answer = tokenizer.decode(output[0], skip_special_tokens=True)

# 12. Вывод ответа (берём текст после prompt)
print("📌 Ответ модели:\n")
print(answer[len(prompt):].strip())


📌 Ответ модели:

1. Pentru persoanele cu dizabilități, adaptăm procesul de recrutare și oferim asistență tehnică pe parcursul interviurilor.
2. La MAIB, fiecare angajat are un plan individual de dezvoltare (IDP) actualizat anual împreună cu managerul direct.
3. Pentru posturile tehnice, procesul include adesea un challenge practic pe care îl poți completa acasă în 72 de ore.
4. Pentru angajații care doresc să se reprofileze, oferim cursuri de recalificare profesională plătite de compan


## Выводы:

Для организации чат-бота, который будет давать строго точный ответ из базы данных на 3-х языках можно применить:

### 1. Просто поиск и отдача из базы

Берём документы на трёх языках, ищем по запросу с помощью векторного поиска, определяем язык и сразу показываем текст из базы. Никаких генераций — значит, точность стопроцентная. и быстро (FAISS)

### 2. Модель с дообучением (fine-tuning) на базе данных
Берём крупную модель, дообучаем её на своих документах и FAQ на трёх языках. Она учится отвечать точно по материалам банка. Минус — требует ресурсов и время, но даёт очень качественные и адаптированные ответы.

### 3. Гибрид: FAQ + поиск + LLM + оператор
Вначале пытаемся ответить из FAQ (просто быстрые ответы). см 1 пункт. Если сложно — ищем в базе и генерируем ответ с помощью LLM. Если и тут сомнения — передаём живому оператору. Самое лучшее качество и поддержка пользователя!

Я лично отдал бы предпочтение 3 варианту, но параллельно дообучал бы модель (пункт 2) на перспективу развития банка.